# flask API

## Objective

The objective of this notebook is to create a Flask API that loads the trained churn prediction model and provides predictions for new customer data.


### Why Flask?

Flask is a lightweight python web framework that can be used to build APIs. In this project, Flask will provide an endpoint that receives customer information and returns a churn prediction.


In [67]:
# Import pandas for data manipulation
import pandas as pd

# Load encoded dataset

df=pd.read_csv("../data/processed/encoded_telco_churn.csv")
df.head()

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,Churn,...,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,TenureGroup_0-12,TenureGroup_13-24,TenureGroup_25-48,TenureGroup_49-72,MonthlyChargeCategory_High,MonthlyChargeCategory_Low,MonthlyChargeCategory_Medium
0,0,0,1,0,1,0,1,29.85,29.85,0,...,0,1,0,1,0,0,0,0,1,0
1,1,0,0,0,34,1,0,56.95,1889.50,0,...,0,0,1,0,0,1,0,0,0,1
2,1,0,0,0,2,1,1,53.85,108.15,1,...,0,0,1,1,0,0,0,0,0,1
3,1,0,0,0,45,0,0,42.30,1840.75,0,...,0,0,0,0,0,1,0,0,0,1
4,0,0,0,0,2,1,1,70.70,151.65,1,...,0,1,0,1,0,0,0,1,0,0


In [68]:
# Seperate input features and target variables

x=df.drop("Churn",axis=1)
y=df["Churn"]

In [69]:
# display feature names used by the model

print(x.columns.tolist())

['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'TotalServices', 'LongTermCustomer', 'MultipleLines_No', 'MultipleLines_No phone service', 'MultipleLines_Yes', 'InternetService_DSL', 'InternetService_Fiber optic', 'InternetService_No', 'OnlineSecurity_No', 'OnlineSecurity_No internet service', 'OnlineSecurity_Yes', 'OnlineBackup_No', 'OnlineBackup_No internet service', 'OnlineBackup_Yes', 'DeviceProtection_No', 'DeviceProtection_No internet service', 'DeviceProtection_Yes', 'TechSupport_No', 'TechSupport_No internet service', 'TechSupport_Yes', 'StreamingTV_No', 'StreamingTV_No internet service', 'StreamingTV_Yes', 'StreamingMovies_No', 'StreamingMovies_No internet service', 'StreamingMovies_Yes', 'Contract_Month-to-month', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Bank transfer (automatic)', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed

In [70]:
# Check how many features model expects

print("Number of features:",x.shape[1])

Number of features: 49


### Understand model input

The model was trained using the encoded dataset, so the API must provide input features in the same structure used during training. Checking the features helps prevent incorrect input from being passed to the model. 

### Observation

The encoded datset contains the features used during model training. The API must maintain the same feature structure and order when making predictions for new customers.

In [71]:
# Impot os to inspect the models directory

import os

# Display files stored in the models folder

print(os.listdir("../models"))

['tuned_random_forest.pkl']


In [72]:
# Import joblib for loading trained model

import joblib
model = joblib.load("../models/tuned_random_forest.pkl")
print(model)


RandomForestClassifier(max_depth=10, min_samples_leaf=2, min_samples_split=5,
                       n_estimators=300, random_state=42)


### Observation

The Trained Random Forest model was successfully loaded and is ready to make predictions.

## Test Model Prediction

Before integrating the model with Flask, a sample prediction is performed to verify that the saved model can correctly accept the encoded feature structure and return a churn prediction.


In [73]:
# Select one customer from the encoded feature dataset

sample = x.iloc[[0]]
sample

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,PaperlessBilling,MonthlyCharges,TotalCharges,TotalServices,...,PaymentMethod_Credit card (automatic),PaymentMethod_Electronic check,PaymentMethod_Mailed check,TenureGroup_0-12,TenureGroup_13-24,TenureGroup_25-48,TenureGroup_49-72,MonthlyChargeCategory_High,MonthlyChargeCategory_Low,MonthlyChargeCategory_Medium
0,0,0,1,0,1,0,1,29.85,29.85,2,...,0,1,0,1,0,0,0,0,1,0


In [74]:
# Generate a churn prediction for the sample customer

sample_pred = model.predict(sample)
print("Churn Prediction:",sample_pred[0])

Churn Prediction: 1


In [75]:
# Get probabilty of each class

sample_prob =model.predict_proba(sample)

print("Probabilty of No Churn:", sample_prob[0][0])
print("Probability of Churn:", sample_prob[0][1])


Probabilty of No Churn: 0.499620513018118
Probability of Churn: 0.5003794869818818


### Observation

The trained Random Forest Model successfully genereated a churn prediction for a sample customer. The model also provided the probability associated with each class, which gives additional information about the prediction.

In [76]:
# Select another customer

sample2 = x.iloc[[1]]
sample_pred2 = model.predict(sample2)
print("Churn prediction:", sample_pred2[0])

Churn prediction: 0


In [77]:
# Display churn probabilty

sample_prob2 = model.predict_proba(sample2)

print("Probability of No Churn:", sample_prob2[0][0])
print("Probabilty of Churn:", sample_prob2[0][1])

Probability of No Churn: 0.9776390380133936
Probabilty of Churn: 0.022360961986606098


### Create Flask API

The Flask API will provide an endpoint that accepts customer data and returns a churn prediction from the trained machine learning model.

In [78]:
# Import Flask commands for creating an API

from flask import Flask, request ,jsonify

# create Flask application

app = Flask(__name__)

In [79]:
# Define the home endpoint

@app.route("/", methods=["GET"])
def home():
    return jsonify({
        "message":"Telecom Customer Churn Prediction API is running"
    })

## Prediction Endpoint

The '/predict' endpoint will receive customer information and return the predicted churn class.

In [80]:
# Define the prediction endpoint

@app.route("/predict",methods=["POST"])
def predict():

    # Get customer data sent as JSON
    data = request.get_json()

    # Convert the input data into a DataFrame
    customer_data = pd.DataFrame([data])

    # Generate churn prediction
    prediction = model.predict(customer_data)

    # Generate churn probability
    probability = model.predict_proba(customer_data)

    # Return the prediction and probabilties as JSON
    return jsonify({
        "churn_prediction": int(prediction[0]),
        "probability_no_churn": float(probability[0][0]),
        "probability_churn": float(probability[0][1])
     }) 

In [81]:
# Display the number of features expected

print("Expected features:", len(x.columns))

Expected features: 49


<!-- # Test API input
 
The API must receive the same features and feature order used during model training. A sample input is created from the encoded feature structure to test the prediction endpoint. -->


## Test API Input

The API must receive the same features and feature order used during model training. A sample input is created from the encoded feature structure to test the prediction endpoint.

In [82]:
# Select one customer from the encoded dataset

test_customer = x.iloc[0].to_dict()

# Display the test input

print(test_customer)

{'gender': 0.0, 'SeniorCitizen': 0.0, 'Partner': 1.0, 'Dependents': 0.0, 'tenure': 1.0, 'PhoneService': 0.0, 'PaperlessBilling': 1.0, 'MonthlyCharges': 29.85, 'TotalCharges': 29.85, 'TotalServices': 2.0, 'LongTermCustomer': 0.0, 'MultipleLines_No': 0.0, 'MultipleLines_No phone service': 1.0, 'MultipleLines_Yes': 0.0, 'InternetService_DSL': 1.0, 'InternetService_Fiber optic': 0.0, 'InternetService_No': 0.0, 'OnlineSecurity_No': 1.0, 'OnlineSecurity_No internet service': 0.0, 'OnlineSecurity_Yes': 0.0, 'OnlineBackup_No': 0.0, 'OnlineBackup_No internet service': 0.0, 'OnlineBackup_Yes': 1.0, 'DeviceProtection_No': 1.0, 'DeviceProtection_No internet service': 0.0, 'DeviceProtection_Yes': 0.0, 'TechSupport_No': 1.0, 'TechSupport_No internet service': 0.0, 'TechSupport_Yes': 0.0, 'StreamingTV_No': 1.0, 'StreamingTV_No internet service': 0.0, 'StreamingTV_Yes': 0.0, 'StreamingMovies_No': 1.0, 'StreamingMovies_No internet service': 0.0, 'StreamingMovies_Yes': 0.0, 'Contract_Month-to-month': 1.

In [83]:
import json

# Convert the sample customer dictonary to JSON format
test_json = json.dumps(test_customer)

print(test_json)

{"gender": 0.0, "SeniorCitizen": 0.0, "Partner": 1.0, "Dependents": 0.0, "tenure": 1.0, "PhoneService": 0.0, "PaperlessBilling": 1.0, "MonthlyCharges": 29.85, "TotalCharges": 29.85, "TotalServices": 2.0, "LongTermCustomer": 0.0, "MultipleLines_No": 0.0, "MultipleLines_No phone service": 1.0, "MultipleLines_Yes": 0.0, "InternetService_DSL": 1.0, "InternetService_Fiber optic": 0.0, "InternetService_No": 0.0, "OnlineSecurity_No": 1.0, "OnlineSecurity_No internet service": 0.0, "OnlineSecurity_Yes": 0.0, "OnlineBackup_No": 0.0, "OnlineBackup_No internet service": 0.0, "OnlineBackup_Yes": 1.0, "DeviceProtection_No": 1.0, "DeviceProtection_No internet service": 0.0, "DeviceProtection_Yes": 0.0, "TechSupport_No": 1.0, "TechSupport_No internet service": 0.0, "TechSupport_Yes": 0.0, "StreamingTV_No": 1.0, "StreamingTV_No internet service": 0.0, "StreamingTV_Yes": 0.0, "StreamingMovies_No": 1.0, "StreamingMovies_No internet service": 0.0, "StreamingMovies_Yes": 0.0, "Contract_Month-to-month": 1.

In [84]:
# Convert the test customer back into DataFrame

test_df = pd.DataFrame([test_customer])

# Generate prediction

prediction = model.predict(test_df)

# Generate prediction probabilities

probability = model.predict_proba(test_df)

print("Churn prediction:", prediction[0])
print("Probability of No Churn:", probability[0][0])
print("Probability of Churn:", probability[0][1])

Churn prediction: 1
Probability of No Churn: 0.499620513018118
Probability of Churn: 0.5003794869818818


<!-- 
### Observation

The model predicted that the sample customer is likely to churn. However, the prediction probabilities are almost evenly divided between the two classes, with approximately 49.96% probability of no churn and 50.04% probability of churn.

This indicates that the model has low confidence in this particular prediction. -->


### Observation

The sample customer data was successfully converted into the feature structure expected by the trained model. The model generated both a churn prediction and the probability of each class, confirming that the test input is suitable for the API.

## Testing the Flask Application

The Flask application is implemented separately in `app/app.py`. The notebook is used to send requests to the running API and verify its responses.

In [94]:
# Run the Flask application
# if __name__ == "__main__":
#     app.run(debug=False,use_reloader= False)

In [92]:
# Import requests for sending HTTP requests
import requests

In [95]:
# Send the GET request to the Flask home endpoint.

response = requests.get("http://127.0.0.1:5000/")

# Display API response
print(response.json())

{'message': 'Telecom Customer Churn Prediction API is running'}


### Observation

The Flask application successfully responded to a GET request through the home endpoint. This confirms that the application is running correctly and can be accessed locally.